In [ ]:
import os
import ast
import glob
import ipaddress

import numpy as np
import pandas as pd
from pandas.errors import EmptyDataError

import hypertor.aug.tcp as tcp_aug
from hypertor.core import util
from hypertor.aug import darkflow

In [37]:
hypertor_data_path = "../../data/"

def normalize_flow_key(src_ip, src_port, dst_ip, dst_port):
    ip1 = int(ipaddress.ip_address(src_ip))
    ip2 = int(ipaddress.ip_address(dst_ip))

    p1 = int(src_port)
    p2 = int(dst_port)

    if (ip1, p1) <= (ip2, p2):
        return f"{ip1}:{p1}-{ip2}:{p2}"
    else:
        return f"{ip2}:{p2}-{ip1}:{p1}"


# RAW

In [ ]:
def csv_handler(dataset: str = "cicdarknet"):
    if dataset.lower() == "cicdarknet":
        base_path = "../../../data/ISCX-Tor-NonTor-2017/Tor"
    elif dataset.lower() == "tcub":
        base_path = "../../../data/darknet-dataset-2020/tor"
    else:
        raise RuntimeError("Wrong dataset")

    categories = sorted([
        d for d in os.listdir(base_path)
        if os.path.isdir(os.path.join(base_path, d))
    ])

    print(f"Found categories: {categories}")

    all_darkflow = []
    all_fp = []

    for category in categories:
        category_path = os.path.join(base_path, category)
        darkflow_path = os.path.join(category_path, "darkflow")
        fp_path = os.path.join(category_path, "fp")

        if not os.path.exists(darkflow_path):
            continue

        darkflow_files = sorted(
            glob.glob(os.path.join(darkflow_path, "*.csv"))
        )

        for file in darkflow_files:
            try:
                df = pd.read_csv(file, encoding='utf-8-sig')
            except EmptyDataError:
                print(f"Skipping empty file: {file}")
                continue
            if df.empty:
                print(f"Skipping empty dataframe: {file}")
                continue
            if df.shape[1] == 0:
                print(f"Skipping file with no columns: {file}")
                continue

            df["label"] = category
            # src_ip:src_port-dst_ip:dst_port
            df["norm_flow_key"] = [
                normalize_flow_key(sip, sport, dip, dport)
                for sip, sport, dip, dport in zip(
                    df["ip_source"],
                    df["port_source"],
                    df["ip_destination"],
                    df["port_destination"]
                )
            ]

            all_darkflow.append(df)

        if os.path.exists(fp_path):
            fp_files = sorted(
                glob.glob(os.path.join(fp_path, "*.csv"))
            )

            for fp_file in fp_files:
                try:
                    df_fp = pd.read_csv(fp_file, encoding='utf-8-sig')
                except EmptyDataError:
                    print(f"Skipping empty file: {fp_file}")
                    continue
                if df_fp.empty:
                    print(f"Skipping empty dataframe: {fp_file}")
                    continue
                if df_fp.shape[1] == 0:
                    print(f"Skipping file with no columns: {fp_file}")
                    continue

                df_fp["norm_flow_key"] = [
                    normalize_flow_key(sip, sport, dip, dport)
                    for sip, sport, dip, dport in zip(
                        df_fp["src_ip"],
                        df_fp["src_port"],
                        df_fp["dst_ip"],
                        df_fp["dst_port"]
                    )
                ]

                all_fp.append(df_fp)

    darkflow_all = pd.concat(all_darkflow, ignore_index=True)
    print("Darkflow merged:", darkflow_all.shape)

    if len(all_fp) > 0:
        fp_all = pd.concat(all_fp, ignore_index=True)
        print("FP merged:", fp_all.shape)
        fp_all = fp_all[["norm_flow_key", "ja4t", "ja4", "ja4l"]]
        fp_all = (
            fp_all
            .sort_values("norm_flow_key")
            .drop_duplicates("norm_flow_key")
        )
        print("FP merged (drop_duplicates):", fp_all.shape)

        fp_all = fp_all.set_index("norm_flow_key")
        final_df = darkflow_all.join(
            fp_all,
            on="norm_flow_key",
            how="left",
        )
    else:
        final_df = darkflow_all

    print("Final merged shape:", final_df.shape)

    return final_df


## CICDarknet

In [ ]:
cic_aug_config = tcp_aug.TCPAugmentationConfig(
    simulate_nagle=False,
    simulate_packet_shift=False,
    simulate_packet_loss=False,
    simulate_packet_duplication=False,
    target_seq_len=30,
    drop_seq_len=3,
)
cic_csv = csv_handler("cicdarknet")

cic_darkflow_raw_file = os.path.join(hypertor_data_path, "cicdarknet", "csv", "darkflow_raw.csv")
os.makedirs(os.path.dirname(cic_darkflow_raw_file), exist_ok=True)
cic_csv.to_csv(cic_darkflow_raw_file, index=False)
print(f"Saved to {cic_darkflow_raw_file}")

cic_filtered_darkflow_raw_file = os.path.join(hypertor_data_path, "cicdarknet", "csv", "darkflow_raw_ja4cleaned.csv")
cic_filtered = cic_csv.dropna(subset=['ja4', 'ja4t', 'ja4l'])
cic_filtered.to_csv(cic_filtered_darkflow_raw_file, index=False)

print(cic_csv.columns)
# print(cic_csv[['flow_key', 'label', 'ja4', 'ja4t', 'ja4l']].head(50))
print(cic_csv.shape)
print(cic_filtered.shape)


Found categories: ['browsing', 'chat', 'email', 'ftp', 'p2p', 'streaming', 'voip']
Skipping empty file: ../../../data/ISCX-Tor-NonTor-2017/Tor/browsing/fp/browsing_ssl_browsing_gateway.csv
Skipping empty file: ../../../data/ISCX-Tor-NonTor-2017/Tor/chat/fp/chat_aimchatgateway.csv
Skipping empty file: ../../../data/ISCX-Tor-NonTor-2017/Tor/chat/fp/chat_facebookchatgateway.csv
Skipping empty file: ../../../data/ISCX-Tor-NonTor-2017/Tor/chat/fp/chat_gate_aim_chat.csv
Skipping empty file: ../../../data/ISCX-Tor-NonTor-2017/Tor/chat/fp/chat_gate_facebook_chat.csv
Skipping empty file: ../../../data/ISCX-Tor-NonTor-2017/Tor/chat/fp/chat_gate_hangout_chat.csv
Skipping empty file: ../../../data/ISCX-Tor-NonTor-2017/Tor/chat/fp/chat_gate_icq_chat.csv
Skipping empty file: ../../../data/ISCX-Tor-NonTor-2017/Tor/chat/fp/chat_gate_skype_chat.csv
Skipping empty file: ../../../data/ISCX-Tor-NonTor-2017/Tor/chat/fp/chat_hangoutschatgateway.csv
Skipping empty file: ../../../data/ISCX-Tor-NonTor-2017/Tor

## TCUB

In [42]:
tcub_aug_config = tcp_aug.TCPAugmentationConfig(
    simulate_nagle=False,
    simulate_packet_shift=False,
    simulate_packet_loss=False,
    simulate_packet_duplication=False,
    target_seq_len=30,
    drop_seq_len=3,
)
tcub_csv = csv_handler("tcub")

tcub_darkflow_raw_file = os.path.join(hypertor_data_path, "tcub", "csv", "darkflow_raw.csv")
os.makedirs(os.path.dirname(tcub_darkflow_raw_file), exist_ok=True)
tcub_csv.to_csv(tcub_darkflow_raw_file, index=False)
print(f"Saved to {tcub_darkflow_raw_file}")

tcub_filtered_darkflow_raw_file = os.path.join(hypertor_data_path, "tcub", "csv", "darkflow_raw_ja4cleaned.csv")
tcub_filtered = tcub_csv.dropna(subset=['ja4', 'ja4t', 'ja4l'])
tcub_filtered.to_csv(tcub_filtered_darkflow_raw_file, index=False)

print(tcub_csv.columns)
print(tcub_csv.shape)
# print(tcub_csv[['flow_key', 'label', 'ja4', 'ja4t', 'ja4l']].head(50))
print(tcub_filtered.shape)


# tcub_aug_config = tcp_aug.TCPAugmentationConfig(
#     simulate_nagle=True,
#     simulate_packet_shift=False,
#     simulate_packet_loss=False,
#     simulate_packet_duplication=False,
#     target_seq_len=30,
#     drop_seq_len=3,
# )
# tcubdata_5d = csv2numpy(tcub_aug_config, "tcub")

Found categories: ['browsing', 'chat', 'email', 'ftp', 'p2p', 'streaming', 'voip']
Skipping empty file: ../../../data/darknet-dataset-2020/tor/browsing/fp/tor-browsing_00001_20200415112319.csv
Skipping empty file: ../../../data/darknet-dataset-2020/tor/chat/fp/tor-chat_00001_20200501114725.csv
Skipping empty file: ../../../data/darknet-dataset-2020/tor/ftp/fp/tor-ftp2_00002_20200427143301.csv
Skipping empty file: ../../../data/darknet-dataset-2020/tor/p2p/fp/tor-p2p_00001_20200417155424.csv
Skipping empty file: ../../../data/darknet-dataset-2020/tor/p2p/fp/tor-p2p_00004_20200430183230.csv
Skipping empty file: ../../../data/darknet-dataset-2020/tor/streaming/fp/tor-video_00003_20200427220153.csv
Darkflow merged: (4507, 25)
FP merged: (1907, 10)
FP merged (drop_duplicates): (1808, 4)
Final merged shape: (4507, 28)
Saved to ../../data/tcub/csv/darkflow_raw.csv
Index(['flow_key', 'ip_source', 'port_source', 'ip_destination',
       'port_destination', 'protocol', 'first_timestamp', 'flow_d

# LOWPTOR

In [45]:
def lowp_csv_handler(dataset: str = "cicdarknet", ratio: float = 0.001):
    if dataset.lower() == "cicdarknet":
        base_path = "../../../data/LOWPTOR/cicdarknet"
    elif dataset.lower() == "tcub":
        base_path = "../../../data/LOWPTOR/tcub"
    else:
        raise RuntimeError("Wrong dataset")

    categories = sorted([
        d for d in os.listdir(base_path)
        if os.path.isdir(os.path.join(base_path, d))
    ])

    print(f"Found categories: {categories}")

    all_darkflow = []
    all_fp = []

    for category in categories:
        category_path = os.path.join(base_path, category)
        darkflow_path = os.path.join(category_path, "darkflow")
        fp_path = os.path.join(category_path, "fp")

        if not os.path.exists(darkflow_path):
            continue

        csv_pattern = f"ewtd-lowptor-{dataset}-{category}-{ratio}.csv"
        darkflow_files = sorted(
            glob.glob(os.path.join(darkflow_path, csv_pattern))
        )

        for file in darkflow_files:
            try:
                df = pd.read_csv(file, encoding='utf-8-sig')
            except EmptyDataError:
                print(f"Skipping empty file: {file}")
                continue
            if df.empty:
                print(f"Skipping empty dataframe: {file}")
                continue
            if df.shape[1] == 0:
                print(f"Skipping file with no columns: {file}")
                continue

            df["label"] = category
            # src_ip:src_port-dst_ip:dst_port
            df["norm_flow_key"] = [
                normalize_flow_key(sip, sport, dip, dport)
                for sip, sport, dip, dport in zip(
                    df["ip_source"],
                    df["port_source"],
                    df["ip_destination"],
                    df["port_destination"]
                )
            ]

            all_darkflow.append(df)

        if os.path.exists(fp_path):
            fp_files = sorted(
                glob.glob(os.path.join(fp_path, csv_pattern))
            )

            for fp_file in fp_files:
                try:
                    df_fp = pd.read_csv(fp_file, encoding='utf-8-sig')
                except EmptyDataError:
                    print(f"Skipping empty file: {fp_file}")
                    continue
                if df_fp.empty:
                    print(f"Skipping empty dataframe: {fp_file}")
                    continue
                if df_fp.shape[1] == 0:
                    print(f"Skipping file with no columns: {fp_file}")
                    continue

                df_fp["norm_flow_key"] = [
                    normalize_flow_key(sip, sport, dip, dport)
                    for sip, sport, dip, dport in zip(
                        df_fp["src_ip"],
                        df_fp["src_port"],
                        df_fp["dst_ip"],
                        df_fp["dst_port"]
                    )
                ]

                all_fp.append(df_fp)

    darkflow_all = pd.concat(all_darkflow, ignore_index=True)
    print("Darkflow merged:", darkflow_all.shape)

    if len(all_fp) > 0:
        fp_all = pd.concat(all_fp, ignore_index=True)
        print("FP merged:", fp_all.shape)
        fp_all = fp_all[["norm_flow_key", "ja4t", "ja4", "ja4l"]]
        fp_all = (
            fp_all
            .sort_values("norm_flow_key")
            .drop_duplicates("norm_flow_key")
        )
        print("FP merged (drop_duplicates):", fp_all.shape)

        fp_all = fp_all.set_index("norm_flow_key")
        final_df = darkflow_all.join(
            fp_all,
            on="norm_flow_key",
            how="left",
        )
    else:
        final_df = darkflow_all

    print("Final merged shape:", final_df.shape)

    return final_df

## CICDarknet

In [50]:
lpcicdarknet_aug_config = tcp_aug.TCPAugmentationConfig(
    simulate_nagle=False,
    simulate_packet_shift=False,
    simulate_packet_loss=False,
    simulate_packet_duplication=False,
    target_seq_len=30,
    drop_seq_len=3,
)
lpcicdarknet_csv = lowp_csv_handler("cicdarknet")

lpcicdarknet_darkflow_raw_file = os.path.join(hypertor_data_path, "lpcicdarknet", "csv", "darkflow_raw.csv")
os.makedirs(os.path.dirname(lpcicdarknet_darkflow_raw_file), exist_ok=True)
lpcicdarknet_csv.to_csv(lpcicdarknet_darkflow_raw_file, index=False)
print(f"Saved to {lpcicdarknet_darkflow_raw_file}")

lpcicdarknet_filtered_darkflow_raw_file = os.path.join(hypertor_data_path, "lpcicdarknet", "csv", "darkflow_raw_ja4cleaned.csv")
lpcicdarknet_filtered = lpcicdarknet_csv.dropna(subset=['ja4', 'ja4t', 'ja4l'])
lpcicdarknet_filtered.to_csv(lpcicdarknet_filtered_darkflow_raw_file, index=False)

print(lpcicdarknet_csv.columns)
# print(lpcicdarknet_csv[['flow_key', 'label', 'ja4', 'ja4t', 'ja4l']].head(50))
print(lpcicdarknet_csv.shape)
print(lpcicdarknet_filtered.shape)


Found categories: ['browsing', 'chat', 'email', 'ftp', 'p2p', 'streaming', 'voip']
Darkflow merged: (348673, 25)
FP merged: (78560, 10)
FP merged (drop_duplicates): (10391, 4)
Final merged shape: (348673, 28)
Saved to ../../data/lpcicdarknet/csv/darkflow_raw.csv
Index(['flow_key', 'ip_source', 'port_source', 'ip_destination',
       'port_destination', 'protocol', 'first_timestamp', 'flow_duration_usec',
       'pkt_dir_seq', 'pkt_len_seq', 'pkt_usec_seq', 'tcp_window_seq',
       'fin_seq', 'syn_seq', 'rst_seq', 'psh_seq', 'ack_seq', 'urg_seq',
       'cwr_seq', 'ece_seq', 'ttl_seq', 'ip_df_seq', 'ip_mf_seq', 'label',
       'norm_flow_key', 'ja4t', 'ja4', 'ja4l'],
      dtype='str')
(348673, 28)
(346658, 28)


## TCUB

In [ ]:
lptcub_aug_config = tcp_aug.TCPAugmentationConfig(
    simulate_nagle=False,
    simulate_packet_shift=False,
    simulate_packet_loss=False,
    simulate_packet_duplication=False,
    target_seq_len=30,
    drop_seq_len=3,
)
lptcub_csv = lowp_csv_handler("tcub")

lptcub_darkflow_raw_file = os.path.join(hypertor_data_path, "lptcub", "csv", "darkflow_raw.csv")
os.makedirs(os.path.dirname(lptcub_darkflow_raw_file), exist_ok=True)
lptcub_csv.to_csv(lptcub_darkflow_raw_file, index=False)
print(f"Saved to {lptcub_darkflow_raw_file}")

lptcub_filtered_darkflow_raw_file = os.path.join(hypertor_data_path, "lptcub", "csv", "darkflow_raw_ja4cleaned.csv")
lptcub_filtered = lptcub_csv.dropna(subset=['ja4', 'ja4t', 'ja4l'])
lptcub_filtered.to_csv(lptcub_filtered_darkflow_raw_file, index=False)

print(lptcub_csv.columns)
# print(lptcub_csv[['flow_key', 'label', 'ja4', 'ja4t', 'ja4l']].head(50))
print(lptcub_csv.shape)
print(lptcub_filtered.shape)

Found categories: ['browsing', 'chat', 'email', 'ftp', 'p2p', 'streaming', 'voip']
Darkflow merged: (350093, 25)
FP merged: (78992, 10)
FP merged (drop_duplicates): (10484, 4)
Final merged shape: (350093, 28)
Saved to ../../data/lptcub/csv/darkflow_raw.csv
Index(['flow_key', 'ip_source', 'port_source', 'ip_destination',
       'port_destination', 'protocol', 'first_timestamp', 'flow_duration_usec',
       'pkt_dir_seq', 'pkt_len_seq', 'pkt_usec_seq', 'tcp_window_seq',
       'fin_seq', 'syn_seq', 'rst_seq', 'psh_seq', 'ack_seq', 'urg_seq',
       'cwr_seq', 'ece_seq', 'ttl_seq', 'ip_df_seq', 'ip_mf_seq', 'label',
       'norm_flow_key', 'ja4t', 'ja4', 'ja4l'],
      dtype='str')
(350093, 28)
(348056, 28)
